# БОРАН — четыре алгоритма шаг за шагом

Финальный проект по AI Fundamentals. Ноутбук показывает ту же цепочку, что и сайт: **HMM → Байес → A\* → MDP**. Каждый шаг: вход → расчёт → выход.

Все вероятности, веса и награды — допущения команды, а не обученная модель.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
from boran import hmm, bayes, astar, mdp, weather, graph, config

## 1. HMM: какое покрытие сейчас

**Скрытые состояния:** сухо, мокро, свежий снег, накат, гололёд. **Наблюдения** получаем из погоды по часам. Сценарий: днём оттепель, ночью мороз.

In [2]:
wx = weather.scenario('thaw_freeze')
for h in wx['past']:
    print(h['time'], f"{h['temp']:+5.1f}°", 'осадки' if h['snow'] >= 0.1 else '')

20:00  +3.0° 
21:00  +4.0° 
22:00  +4.0° осадки
23:00  +3.0° осадки
00:00  +2.0° 
01:00  +1.0° 
02:00  +0.0° 
03:00  -2.0° 
04:00  -4.0° 
05:00  -6.0° 
06:00  -8.0° 
07:00  -9.0° 


In [3]:
obs = hmm.observe(wx['past'])
print('наблюдения:', [hmm.OBS[o] for o in obs])

наблюдения: ['thaw_dry', 'thaw_dry', 'thaw_wet', 'thaw_wet', 'thaw_dry', 'zero_cross', 'zero_cross', 'frost_dry', 'frost_dry', 'frost_dry', 'frost_dry', 'frost_dry']


**Forward-фильтр** считает P(состояние | все наблюдения до этого часа):

$$b_t(j) \propto e_j(o_t)\sum_i b_{t-1}(i)\,a_{ij}$$

In [4]:
beliefs = hmm.forward(obs)
print('час   ' + '  '.join(f'{s:>6}' for s in hmm.STATES))
for h, b in zip(wx['past'], beliefs):
    print(h['time'], '  '.join(f'{p:6.2f}' for p in b))

час      dry     wet    snow  packed     ice
20:00   0.84    0.07    0.02    0.05    0.02
21:00   0.91    0.07    0.01    0.01    0.01
22:00   0.29    0.60    0.09    0.01    0.01
23:00   0.06    0.87    0.04    0.01    0.02
00:00   0.42    0.52    0.01    0.01    0.04
01:00   0.22    0.29    0.04    0.02    0.44
02:00   0.09    0.14    0.04    0.03    0.71
03:00   0.16    0.00    0.03    0.09    0.72
04:00   0.19    0.00    0.03    0.13    0.65
05:00   0.22    0.00    0.03    0.16    0.58
06:00   0.24    0.00    0.03    0.19    0.53
07:00   0.27    0.00    0.03    0.21    0.49


**Витерби** находит самую вероятную историю состояний целиком.

In [5]:
path, logp = hmm.viterbi(obs)
print('путь Витерби:', ' → '.join(hmm.STATES[i] for i in path))
print('log P =', round(logp, 3))
now = beliefs[-1]
print('сейчас:', {s: round(p, 3) for s, p in zip(hmm.STATES, now)})

путь Витерби: dry → dry → wet → wet → wet → ice → ice → ice → ice → ice → ice → ice
log P = -18.863
сейчас: {'dry': 0.266, 'wet': 0.002, 'snow': 0.035, 'packed': 0.206, 'ice': 0.492}


## 2. Наивный Байес: опасен ли участок

Prior берём из HMM: P(скользко) = P(гололёд) + 0.4·P(накат) + 0.15·P(снег) + 0.1·P(мокро). Затем учитываем признаки улицы.

In [6]:
prior = bayes.prior_from_hmm(now)
print('prior P(скользко) =', round(prior, 4))
print('P(признак | скользко), P(признак | не скользко):')
for f, (a, b) in bayes.LIKELIHOOD.items():
    print(f'  {f:10} {a:.2f}  {b:.2f}')

prior P(скользко) = 0.5799
P(признак | скользко), P(признак | не скользко):
  shade      0.55  0.30
  open       0.50  0.35
  uncleaned  0.70  0.40
  stop       0.25  0.10
  bridge     0.10  0.03


In [7]:
street = {'shade': True, 'open': False, 'uncleaned': True, 'stop': False, 'bridge': False}
res = bayes.explain(prior, street)
for s in res['steps']:
    print(f"{s['feature']:10} {'есть' if s['present'] else 'нет ':4}  → P = {s['after']:.4f}")
print('posterior P(скользко | признаки) =', res['posterior'])

shade      есть  → P = 0.7168
open       нет   → P = 0.6607
uncleaned  есть  → P = 0.7731
stop       нет   → P = 0.7395
bridge     нет   → P = 0.7248
posterior P(скользко | признаки) = 0.7248


In [8]:
main_road = {'shade': False, 'open': True, 'uncleaned': False, 'stop': False, 'bridge': False}
print('двор в тени:      ', round(bayes.posterior(prior, street), 3))
print('чищеный проспект: ', round(bayes.posterior(prior, main_road), 3))

двор в тени:       0.725
чищеный проспект:  0.329


## 3. A\*: безопасный маршрут

Маленький граф из 5 перекрёстков. Прямая дорога 0→1→2 короче (280 м), но скользкая. Обход 0→3→4→2 длиннее (440 м), но чистый.

Стоимость участка = длина × (1 + α·P(скользко) + β·ветер), α = 4. Эвристика h — расстояние по прямой, она никогда не завышает стоимость.

In [9]:
nodes = [[71.400, 51.100], [71.402, 51.100], [71.404, 51.100], [71.401, 51.1012], [71.403, 51.1012]]
def edge(u, v, length):
    return {'u': u, 'v': v, 'len': length, 'mask': 0, 'warm': False, 'hw': 'footway', 'p': [nodes[u], nodes[v]]}
edges = [edge(0, 1, 140), edge(1, 2, 140), edge(0, 3, 150), edge(3, 4, 140), edge(4, 2, 150)]
city = graph.City({'bbox': [51.09, 71.39, 51.11, 71.41], 'nodes': nodes, 'edges': edges, 'buildings': [], 'water': {'lines': [], 'polys': []}, 'stops': [], 'warm': []})
risks = [0.9, 0.9, 0.05, 0.05, 0.05]
for e, r in zip(edges, risks):
    print(f"{e['u']}–{e['v']}: {e['len']} м, P = {r}, стоимость = {astar.edge_cost(e, r, 0.0):.0f}")

0–1: 140 м, P = 0.9, стоимость = 644
1–2: 140 м, P = 0.9, стоимость = 644
0–3: 150 м, P = 0.05, стоимость = 180
3–4: 140 м, P = 0.05, стоимость = 168
4–2: 150 м, P = 0.05, стоимость = 180


In [10]:
short = astar.search(city, 0, 2, risks, 0.0, safe=False)
safe = astar.search(city, 0, 2, risks, 0.0, safe=True)
print('короткий:   ', short['nodes'], 'длина', short['cost'], 'раскрыто узлов', len(short['order']))
print('безопасный: ', safe['nodes'], 'стоимость', round(safe['cost'], 1), 'раскрыто узлов', len(safe['order']))
for name, r in (('короткий', short), ('безопасный', safe)):
    d = astar.describe(city, r, risks, 0.0)
    print(f"{name}: {d['length']} м, {d['minutes']} мин, риск упасть {d['fall'] * 100:.2f}%")

короткий:    [0, 1, 2] длина 280.0 раскрыто узлов 3
безопасный:  [0, 3, 4, 2] стоимость 528.0 раскрыто узлов 4
короткий: 280 м, 4.7 мин, риск упасть 1.25%
безопасный: 440 м, 6.0 мин, риск упасть 0.11%


### 3b. Пять алгоритмов поиска на настоящих улицах Астаны

Граф пешеходных улиц всей Астаны из OpenStreetMap. Маршрут: кампус Qairu → Хан Шатыр. Одинаковые точки, одинаковый граф, разные стратегии.

- **BFS** — очередь, раскрывает слоями, минимум участков.
- **DFS** — стек, уходит вглубь, путь не оптимален.
- **UCS** (Дейкстра) — очередь с приоритетом по стоимости g.
- **Greedy** — приоритет по эвристике h (расстояние по прямой).
- **A\*** — приоритет f = g + h.

In [11]:
from boran import pipeline, search
city = graph.get_city()
print('перекрёстков:', len(city.nodes), '| участков улиц:', len(city.edges))
an = pipeline.analyse(city, weather.scenario('thaw_freeze'))
start = city.nearest_node(71.417377, 51.089187)
goal = city.nearest_node(71.4036, 51.1323)
res = search.compare(city, an, start, goal)
print(f"{'алгоритм':8} {'раскрыто':>9} {'участков':>9} {'длина, км':>10} {'риск, %':>8} {'стоимость':>10} {'мс':>6}")
for r in res:
    print(f"{r['algo']:8} {r['expanded']:9d} {r['steps']:9d} {r['length'] / 1000:10.1f} {r['fall'] * 100:8.1f} {r['cost']:10.0f} {r['ms']:6.1f}", '← оптимально' if r['best_cost'] else '')

перекрёстков: 31904 | участков улиц: 45103
алгоритм  раскрыто  участков  длина, км  риск, %  стоимость     мс
bfs          19661        86        6.7     13.8      18869   17.4 
dfs          21152      5692      422.8    100.0    1239257   19.3 
ucs          13143       106        6.8      9.9      14593   24.1 ← оптимально
greedy         142       134        6.6     16.1      20785    0.4 
astar        10337       106        6.8      9.9      14593   26.6 ← оптимально


UCS и A\* находят один и тот же самый дешёвый путь, но A\* раскрывает меньше перекрёстков благодаря эвристике. BFS находит путь с наименьшим числом участков, но он длиннее и опаснее. Greedy думает быстрее всех, но его путь дороже. DFS находит путь, но это огромный крюк по всему району.

## 4. MDP: что делать сейчас

**Состояния:** спокойно / снег / буран. **Действия:** идти, ждать 20 минут, транспорт. Награда — минус «минуты потерь»: время + холод + риск упасть.

$$V(s)=\max_a\Big[R(s,a)+\gamma\sum_{s'}P(s'\mid s,a)\,V(s')\Big]$$

Сценарий: буран 18 м/с, по прогнозу стихает. Идти пешком 30 минут.

In [12]:
bl = weather.scenario('blizzard')
sev_now, sev_next = mdp.severity(bl['now']), mdp.severity(bl['ahead'][2])
level, trend = mdp.level(sev_now), mdp.trend_from_forecast(sev_now, sev_next)
print('тяжесть сейчас', round(sev_now, 2), '→ через 3 часа', round(sev_next, 2))
print('уровень:', level, '| тренд:', trend)
print('P(следующий уровень):', dict(zip(mdp.LEVELS, mdp.TREND_P[trend][level])))

тяжесть сейчас 0.92 → через 3 часа 0.49
уровень: blizzard | тренд: ease
P(следующий уровень): {'calm': 0.15, 'snow': 0.5, 'blizzard': 0.35}


In [13]:
inp = {'walk_min': 30.0, 'walk_risk': 0.027, 'ride_min': 6.0, 'access_min': 6.0, 'access_risk': 0.007, 'cold_factor': mdp.cold_factor(bl['now']['apparent'])}
res = mdp.solve(inp, level, trend)
print('cold_factor =', round(inp['cold_factor'], 3))
print('уровень     ' + '  '.join(f'{a:>8}' for a in mdp.ACTIONS))
for s in mdp.LEVELS:
    q = res['table'][s]
    print(f'{s:10}  ' + '  '.join(f"{q.get(a, float('nan')):8.1f}" for a in mdp.ACTIONS), ' ← сейчас' if s == level else '')
print('лучшее действие:', res['best'])

cold_factor = 0.533
уровень           go      wait   transit
calm           -47.6     -53.5     -34.3 
snow           -59.4     -58.9     -44.0 
blizzard       -91.8     -69.0     -79.4  ← сейчас
лучшее действие: wait


In [14]:
calm = mdp.solve(dict(inp, cold_factor=0.1, walk_min=8.0, walk_risk=0.01), 'calm', 'steady')
print('короткий путь в спокойную погоду:', calm['q'], '→', calm['best'])

короткий путь в спокойную погоду: {'go': -9.4, 'transit': -28.7, 'wait': -29.2} → go


## Итог

0. Пять алгоритмов поиска (BFS, DFS, UCS, Greedy, A\*) сравнены на реальном графе улиц Астаны.
1. HMM по погоде восстановил скрытое покрытие: после оттепели и ночного мороза вероятнее всего гололёд.
2. Байес превратил это в риск для каждой улицы: двор в тени опаснее чищеного проспекта.
3. A\* выбрал путь длиннее, но безопаснее.
4. MDP посоветовал переждать буран, потому что он стихает.